In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import mutual_info_classif


print("Loading Brazilian dataset...")

dataset = load_dataset(
    "joelniklaus/brazilian_court_decisions",
    split="train[:2000]"
)

df_br = pd.DataFrame(dataset)

print("Dataset loaded successfully.")
print("Shape:", df_br.shape)


# -----------------------------------------
# TEXT CLEANING
# -----------------------------------------

def clean_brazilian_text(text):

    text = str(text).lower()
    text = re.sub(r"[^a-zà-ÿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


text_columns = [
    "ementa_text",
    "decision_description",
    "judgment_text",
    "unanimity_text"
]

for col in text_columns:
    df_br[col] = df_br[col].fillna("").apply(clean_brazilian_text)


# -----------------------------------------
# TARGET REPRESENTATION
# -----------------------------------------

print("\nVectorizing decision_description...")

tfidf_target = TfidfVectorizer(max_features=500)

target_vectors = tfidf_target.fit_transform(df_br["decision_description"]).toarray()

# convert to pseudo-label using dominant topic
target_labels = np.argmax(target_vectors, axis=1)


# -----------------------------------------
# CANDIDATE COLUMNS
# -----------------------------------------

candidate_columns = [
    "orgao_julgador",
    "publish_date",
    "judge_relator",
    "ementa_text",
    "judgment_text",
    "judgment_label",
    "unanimity_text",
    "unanimity_label"
]

scores = {}

print("\nCalculating importance scores...\n")


for col in candidate_columns:

    column_data = df_br[col].fillna("").astype(str)

    # TEXT COLUMN → TF-IDF
    if col in ["ementa_text", "judgment_text", "unanimity_text"]:

        tfidf = TfidfVectorizer(max_features=200)
        X = tfidf.fit_transform(column_data).toarray()

        mi_scores = mutual_info_classif(X, target_labels)

        score = np.mean(mi_scores)

    else:
        encoder = LabelEncoder()
        X = encoder.fit_transform(column_data)

        score = mutual_info_classif(
            X.reshape(-1,1),
            target_labels
        )[0]

    scores[col] = score


# -----------------------------------------
# SORT RESULTS
# -----------------------------------------

ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("="*60)
print("COLUMN IMPORTANCE SCORES")
print("="*60)

for col,score in ranking:
    print(f"{col:20s} : {score:.4f}")


# -----------------------------------------
# SELECT TOP COLUMNS
# -----------------------------------------

TOP_K = 5

selected_metadata = [col for col,score in ranking[:TOP_K]]

print("\nSelected Metadata Columns for DAERM:")
print(selected_metadata)

Loading Brazilian dataset...


Using the latest cached version of the dataset since joelniklaus/brazilian_court_decisions couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\natra\.cache\huggingface\datasets\joelniklaus___brazilian_court_decisions\default\0.0.0\e937c2db8eab109cafc4f5279a396957d38251c5 (last modified on Tue Dec  9 22:25:31 2025).


Dataset loaded successfully.
Shape: (2000, 10)

Vectorizing decision_description...

Calculating importance scores...

COLUMN IMPORTANCE SCORES
orgao_julgador       : 0.7331
unanimity_label      : 0.6147
judge_relator        : 0.3507
judgment_label       : 0.0989
ementa_text          : 0.0748
publish_date         : 0.0470
judgment_text        : 0.0140
unanimity_text       : 0.0087

Selected Metadata Columns for DAERM:
['orgao_julgador', 'unanimity_label', 'judge_relator', 'judgment_label', 'ementa_text']
